In [1]:
%pip install -q "transformers>=4.51,<5"

from pathlib import Path
import random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out = Path("lab_outputs")
out.mkdir(exist_ok=True)
print("Device:", device)

model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Device: cpu


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
# Expanded 18-pair training dataset (strictly disjoint from the 10 held-out topics)
train_examples = [
    ("graphs", "Sketch a graph, then mark a path through three vertices."),
    ("Python loops", "Trace a loop by writing the counter after each step."),
    ("fractions", "Draw equal parts and label the fraction you shaded."),
    ("sets", "Draw a Venn diagram and label the intersection."),
    ("sorting", "Sort five values by hand and record each swap."),
    ("probability", "List the outcomes of two coin tosses before counting."),
    ("stacks", "Draw a vertical box and trace push and pop operations step by step."),
    ("queues", "Line up five numbers and track the front and rear pointers after each dequeue."),
    ("linked lists", "Draw nodes as boxes with arrows and update next pointers before deleting a node."),
    ("hash tables", "Compute the modulo index for five keys and note how collisions resolve."),
    ("SQL joins", "Write two small four-row tables and highlight matching keys before joining."),
    ("derivatives", "Sketch the curve and estimate the tangent slope at two points."),
    ("linear equations", "Substitute your solved variable back into both equations to verify."),
    ("Boolean algebra", "Build a four-row truth table to verify both sides of the identity."),
    ("regular expressions", "Test your pattern against three matching strings and three edge cases."),
    ("CSS flexbox", "Draw the main axis and cross axis arrows before setting justify-content."),
    ("Git branching", "Sketch commits as circles on a timeline before running merge or rebase."),
    ("neural networks", "Write the input, weight, and output dimensions beside each layer."),
]

tuned_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tuned_model.requires_grad_(False)
tuned_model.transformer.h[-1].requires_grad_(True)
opt = torch.optim.AdamW(tuned_model.transformer.h[-1].parameters(), lr=5e-5)

for epoch in range(6):
    tuned_model.train()
    total_loss = 0.0
    for topic, answer in train_examples:
        prefix = tokenizer.encode(f"Topic: {topic}\nStudy tip:")
        target = tokenizer.encode(" " + answer) + [tokenizer.eos_token_id]
        ids = torch.tensor([prefix + target], device=device)
        labels = torch.tensor([[-100] * len(prefix) + target], device=device)
        opt.zero_grad()
        loss = tuned_model(input_ids=ids, labels=labels).loss
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}: avg_loss = {round(total_loss / len(train_examples), 3)}")

tuned_model.save_pretrained(out / "study_tip_gpt2")
tokenizer.save_pretrained(out / "study_tip_gpt2")
print("Saved Capstone model to lab_outputs/study_tip_gpt2\n")

# Evaluate on 10 Held-Out Topics (Manual Page 15)
def generate_tip(net, topic):
    prompt = f"Topic: {topic}\nStudy tip:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        tokens = net.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=45,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(tokens[0, inputs.input_ids.size(1):], skip_special_tokens=True).strip()

held_out_10 = [
    "recursion",
    "matrices",
    "binary search",
    "dynamic programming",
    "trees",
    "normalization",
    "pointers",
    "eigenvalues",
    "APIs",
    "backpropagation",
]

tuned_model.eval()
for i, topic in enumerate(held_out_10, 1):
    print(f"{i:02d}. TOPIC: {topic}")
    print("    BASE   :", generate_tip(base_model, topic))
    print("    ADAPTED:", generate_tip(tuned_model, topic))

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1: avg_loss = 5.388
Epoch 2: avg_loss = 4.955
Epoch 3: avg_loss = 4.826
Epoch 4: avg_loss = 4.535
Epoch 5: avg_loss = 4.317
Epoch 6: avg_loss = 4.033
Saved Capstone model to lab_outputs/study_tip_gpt2

01. TOPIC: recursion
    BASE   : Use the "recursive" keyword to avoid using recursive functions. This is a great way of avoiding having too many variables in your program, and it's also an excellent tool for debugging problems that you might have with other programs
    ADAPTED: Use the following to create a new table and save it.
02. TOPIC: matrices
    BASE   : Use the matrix to create a new row. The first column is your current position in this table, and you can use it as an index for other rows that are not currently available (e., columns 1 through 5). For
    ADAPTED: Use the matrix as a starting point.
03. TOPIC: binary search
    BASE   : Use the "search" button to find your favorite words. If you're not sure what word is in question, use a different keyword or phrase tha